# Example 1.6 / 1: Structure: One Section per Component

Companion notebook for **1.6 Writing Readable Models** in the AMPL lab Documentation, section "Structure: One Section per Component". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Structure: One Section per Component

Order the model file by component, each in its own commented section: sets, parameters and
variables first, then the objective and the constraints. A short header says what the model is
for. (The style guide lists constraints before objectives; the AMPL book usually puts the
objective first. Either works, since each part only uses names declared above it.)

In `data.py` and `run.py`, `ampl` is an `AMPL` object that has already read `model.mod`. The
data comes from Python structures: a list for the set, `ampl.set["PRODUCTS"] = [...]`, and a
dictionary keyed by the set's members for each parameter, `ampl.param["rate"] = {...}`.
`run.py` solves with `ampl.solve(solver="highs")`, checks `ampl.solve_result` before using the
results, and reads them into Python: `to_dict()` returns the variable's values as a dictionary,
and `value()` the objective's value:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
# Production planning: choose how much of each product to make
# to maximize profit within the available hours.

### SETS
set PRODUCTS;                          # products that can be made

### PARAMETERS
param rate {PRODUCTS} > 0;             # tons produced per hour
param profit_per_ton {PRODUCTS};       # profit per ton sold
param max_sales {PRODUCTS} >= 0;       # most that can be sold, in tons
param hours_available >= 0;            # production hours in the period

### VARIABLES
var Production {p in PRODUCTS} >= 0, <= max_sales[p];   # tons to make

### OBJECTIVE
maximize TotalProfit:
    sum {p in PRODUCTS} profit_per_ton[p] * Production[p];

### CONSTRAINTS
subject to HoursLimit:
    sum {p in PRODUCTS} Production[p] / rate[p] <= hours_available;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PRODUCTS"] = ["bands", "coils"]
ampl.param["hours_available"] = 40
ampl.param["rate"] = {"bands": 200, "coils": 140}
ampl.param["profit_per_ton"] = {"bands": 25, "coils": 30}
ampl.param["max_sales"] = {"bands": 6000, "coils": 4000}

In [5]:
# run.py
ampl.solve(solver="highs")
if ampl.solve_result != "solved":
    raise RuntimeError(f"not solved: {ampl.solve_result}")

print(ampl.var["Production"].to_dict())
print(ampl.obj["TotalProfit"].value())

HiGHS 1.14.0: optimal solution; objective 192000
0 simplex iterations
0 barrier iterations
{'bands': 6000, 'coils': 1400}
192000.0


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`. The data file uses the same names, and
`display` prints the result:

In [6]:
%%writefile data.dat
set PRODUCTS := bands coils;
param hours_available := 40;
param:  rate  profit_per_ton  max_sales :=
  bands  200        25           6000
  coils  140        30           4000 ;

Writing data.dat


In [7]:
%%ampl_eval
reset;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
solve;
display Production;
display TotalProfit;

HiGHS 1.14.0: optimal solution; objective 192000
0 simplex iterations
0 barrier iterations
Production [*] :=
bands  6000
coils  1400
;

TotalProfit = 192000

